# Initialization

In [89]:
# !pip install dlib pillow requests scipy tqdm
# Cell 2: Import necessary libraries
import scipy.stats  # For percentile calculations (normal distribution)
import os
import bz2
import requests
import numpy as np
import dlib
import PIL.Image
import scipy.ndimage
import matplotlib.pyplot as plt
import cv2
import numpy as np
from PIL import Image
from IPython.display import display, clear_output
import sys
from blur_wavelet import blur_detect as hwt_blur_detect
from collections import defaultdict
from scipy import stats
from natsort import natsorted
import shutil
from tqdm import tqdm
landmarks_model_path = "data_root/cache/ffhq/shape_predictor_68_face_landmarks.dat"
from IPython.display import clear_output


def pil_to_cv2(pil_img):
    """
    Convert a PIL Image to an OpenCV image (NumPy array in BGR format).
    Returns a new array; does not modify the original.
    """
    img = np.array(pil_img)
    if img.ndim == 2:  # Grayscale
        return img.copy()
    elif img.shape[2] == 4:  # RGBA
        return cv2.cvtColor(img, cv2.COLOR_RGBA2BGRA)
    else:  # RGB
        return cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

def is_hwt_blurry(img=None, img_path=None, threshold=35, min_zero=0.0000):

    if img is None and img_path is not None:
        processed_img = cv2.imread(img_path)
    elif isinstance(img, Image.Image):
        processed_img = pil_to_cv2(img)
    elif isinstance(img, np.ndarray):
        processed_img = img.copy()
    else:
        raise ValueError("You must provide either img or img_path.")
    per, blurext = hwt_blur_detect(processed_img, threshold)
    is_blur = per <= min_zero
    
    print(f"Blur detection percentage: {per:.5f}, is blurry: {is_blur}")
    return is_blur

    
def is_blurry(img=None, img_path=None, threshold=100):
    """
    Check if the image is blurry using Laplacian variance.
    Preserves the original image object.
    """
    if img is None and img_path is not None:
        processed_img = cv2.imread(img_path)
    elif isinstance(img, Image.Image):
        processed_img = pil_to_cv2(img)
    elif isinstance(img, np.ndarray):
        processed_img = img.copy()
    else:
        raise ValueError("You must provide either img or img_path.")

    gray = cv2.cvtColor(processed_img, cv2.COLOR_BGR2GRAY)
    variance = cv2.Laplacian(gray, cv2.CV_64F).var()
    print(f"Laplacian variance: {variance:.2f}")
    return variance < threshold

def is_bad_fsb_lighting(img=None, img_path=None, person_stats=None):
    if img is None and img_path is not None:
        processed_img = cv2.imread(img_path)
    elif isinstance(img, Image.Image):
        processed_img = pil_to_cv2(img)
    elif isinstance(img, np.ndarray):
        processed_img = img.copy()
    else:
        raise ValueError("You must provide either img or img_path.")
    
    results = process_fsb_lighting_face_image(processed_img)
    
    if results is None:
        print("❌⚠️  No face detected.")
        return True # bad lighting if no face detected
    fsb = results['fsb']
    
    
    print(f"FSB (Face Specific Brightness): {fsb:.2f}")
    
    if person_stats is not None:
        lower_bound, upper_bound = get_fsb_bounds(person_stats['race'], person_stats['gender'])
        # print(f"FSB bounds for {person_stats['race']} {person_stats['gender']}: {lower_bound:.2f} - {upper_bound:.2f}")
        if lower_bound <= fsb <= upper_bound:
            # print(" FSB is inside the acceptable range for this demographic (unexpected).")
            return False
        else:
            # print("⚠️ FSB is outside the acceptable range for this demographic (expected).")
            return True

    # print(f"FSB (Face Specific Brightness): {fsb:.2f}")
    
    




In [90]:

def get_image_paths(directory):
    """Return all image file paths in the given directory.
    
    Args:
        directory (str): Path to the directory containing images.
        
    Returns:
        list: List of full paths to image files.
    """
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.webp'}
    image_paths = []
    
    for root, _, files in os.walk(directory):
        for file in files:
            # Check if file has an image extension (case insensitive)
            if os.path.splitext(file)[1].lower() in image_extensions:
                full_path = os.path.join(root, file)
                image_paths.append(full_path)
    
    return image_paths


def get_random_image_path(image_folder):
    """
    Returns a randomly selected image file path from the given folder.
    
    Args:
        image_folder (str): Path to the folder containing images.
        
    Returns:
        str: Full path to a randomly selected image file.
        
    Raises:
        ValueError: If no image files are found in the folder.
    """
    # Common image file extensions
    image_extensions = ['.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp']
    
    # Get all files in the folder that have image extensions (case-insensitive)
    image_files = [
        f for f in os.listdir(image_folder) 
        if os.path.splitext(f)[1].lower() in image_extensions
    ]
    
    if not image_files:
        raise ValueError(f"No image files found in {image_folder}")
    
    # Select a random image file
    random_image = random.choice(image_files)
    
    # Return the full path
    return os.path.join(image_folder, random_image)

In [91]:
# ==========================================
# 1. PATH CONFIGURATION - CRITICAL FIX
# ==========================================
BASE_DIR = "/home/nessessence/mnt_tl_vision16/home/nessessence/uul"
FACE_BRIGHTNESS_DIR = os.path.join(BASE_DIR, "FaceBrightness")
FACE_PARSING_DIR = os.path.join(FACE_BRIGHTNESS_DIR, "face_parsing")
# Clear conflicting paths and add correct ones
sys.path = [p for p in sys.path if "uul" not in p]
sys.path.insert(0, FACE_PARSING_DIR)
sys.path.insert(0, FACE_BRIGHTNESS_DIR)

print("✅ Python path configured:")
for p in sys.path[:2]:
    print(f"→ {p}")
# ==========================================
# 2. IMPORTS WITH VERIFICATION
# ==========================================
try:
    # Verify we're importing from correct location
    import face_parsing.test as face_test
    print(f"✓ Importing from: {face_test.__file__}")
    if not face_test.__file__.startswith(FACE_PARSING_DIR):
        raise ImportError("Wrong test.py being imported!")
    from face_parsing.test import evaluate
    from demographic_face_analyze import without_beard_region
    print("✓ All imports successful!")
    
except Exception as e:
    print(f"\n❌ Import Error: {e}")
    print("\n🔍 Debugging Info:")
    # Check test.py contents
    test_py_path = os.path.join(FACE_PARSING_DIR, "test.py")
    if os.path.exists(test_py_path):
        print(f"\nContents of {test_py_path}:")
        with open(test_py_path, 'r') as f:
            print(f.read(500))  # Show first 500 chars
        
    print("\n💡 Solutions:")
    print("1. Temporary rename conflicting file:")
    print(f"   !mv {BASE_DIR}/test.py {BASE_DIR}/test.py.BAK")
    print("2. Or use absolute import:")
    print("   from FaceBrightness.face_parsing.test import evaluate")
    raise

# ==========================================
# 3. CORE ANALYSIS FUNCTIONS
# ==========================================
def weights_calc(data):
    """Calculate weights for brightness values"""
    num_list = list(data.values())
    total = sum(num_list)
    return np.array([num/total for num in num_list])
def calculate_fsb(image):
    """Calculate Face Skin Brightness (FSB) as per paper"""
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image
    return np.mean(gray)
def calculate_bim(image):
    """Calculate Brightness Information Metric (BIM)"""
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Calculate histogram
    hist = cv2.calcHist([image], [0], None, [256], [0,256])
    hist /= hist.sum()  # Normalize
    
    level = np.arange(256)
    avg_brightness = np.sum(level * hist.flatten())
    bim = np.sum(np.abs(level - avg_brightness) * hist.flatten())
    return bim


def without_beard_region(image, mask, position=False, flatten=True):
    """
    Extract the upper face region (above the bottom of the nose).
    
    Args:
        image: Input BGR or grayscale image.
        mask: Face parsing mask.
        position: If True, also return pixel coordinates.
        flatten: If True, return 1D pixel values. If False, return a 2D (or 3D) masked image.
    
    Returns:
        Either a 1D array of pixel intensities or a masked image,
        and optionally pixel coordinates.
    """
    assert ((len(image.shape) == 3) or (len(image.shape) == 2)), f"Expect a gray or colored image, but got shape: {image.shape}"
    
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        h, w = gray.shape
    else:
        gray = image
        h, w = image.shape

    # Resize mask to match image size if needed
    if mask.shape[:2] != (h, w):
        mask = cv2.resize(mask.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
    else:
        mask = mask.astype(np.uint8)

    # Find all skin pixels (label 1)
    face_pos = np.where(mask == 1)

    # Find nose region (label 10)
    nose = np.where(mask == 10)
    if len(nose[0]) == 0:
        print("Nose not found in mask.")
        return None

    b_lim = nose[0][-1]  # bottom Y of the nose

    # Keep only skin pixels above bottom of nose
    mask_upper = np.zeros_like(mask, dtype=np.uint8)
    for y, x in zip(face_pos[0], face_pos[1]):
        if y < b_lim:
            mask_upper[y, x] = 1

    # Apply mask to grayscale image
    if flatten:
        pixel_values = gray[mask_upper == 1]
        return (pixel_values, np.where(mask_upper == 1)) if position else pixel_values
    else:
        masked_image = cv2.bitwise_and(gray, gray, mask=mask_upper)
        return (masked_image, np.where(mask_upper == 1)) if position else masked_image
    
def process_fsb_lighting_face_image(img):
    """Full processing pipeline for single image"""
    # Load image
    # img = cv2.imread(img_path)
    # if img is None:
    #     raise ValueError(f"Could not read image at {img_path}")
    # Get face mask
    mask = evaluate([img])[0]  # evaluate expects a list
    # print(f"Mask shape: {mask.shape}, unique values: {np.unique(mask)}")
    # Extract upper face region (excluding eyes, nose, lips etc.)
    upper_face = without_beard_region(img, mask)
    if upper_face is None:
        return None
    # print(f"Upper face : {upper_face}")
    
    # Calculate both metrics
    fsb = calculate_fsb(upper_face)
    bim = calculate_bim(upper_face)
    
    # for visualization, we can get the full masked image
    upper_face = without_beard_region(img, mask, flatten=False)  # Get full masked image
    
    
    return {
        'original': img,
        'mask': mask,
        'upper_face': upper_face,
        'fsb': fsb,
        'bim': bim
    }
# ==========================================
# 4. VISUALIZATION FUNCTIONS
# ==========================================
def display_results(results):
    """Display processing results with matplotlib"""
    # clear_output(wait=True)
    print(f"-FSB (Face Skin Brightness): {results['fsb']:.1f}/255")
    
    # Create figure
    plt.figure(figsize=(18, 5))
    
    # Original Image
    plt.subplot(1, 4, 1)
    plt.imshow(cv2.cvtColor(results['original'], cv2.COLOR_BGR2RGB))
    plt.title("Original Image")
    plt.axis('off')
    
    # Face Mask
    plt.subplot(1, 4, 2)
    plt.imshow(results['mask'], cmap='tab20')
    plt.title("Face Parsing Mask")
    plt.axis('off')
    
    # Upper Face Region
    plt.subplot(1, 4, 3)
    plt.imshow(cv2.cvtColor(results['upper_face'], cv2.COLOR_BGR2RGB))
    plt.title("Upper Face Region")
    plt.axis('off')
    
    # Metrics display
    plt.subplot(1, 4, 4)
    plt.text(0.1, 0.7, f"FSB: {results['fsb']:.2f}\nBIM: {results['bim']:.2f}", 
             fontsize=12, bbox=dict(facecolor='white', alpha=0.5))
    plt.axis('off')
    plt.title("Brightness Metrics")
    
    plt.tight_layout()
    plt.show()
    
    # Brightness Histogram
    # gray = cv2.cvtColor(results['upper_face'], cv2.COLOR_BGR2GRAY) if len(results['upper_face'].shape) == 3 else results['upper_face']
    # plt.figure(figsize=(10, 4))
    # plt.hist(gray.ravel(), 256, [0,256])
    # plt.title("Pixel Brightness Distribution")
    # plt.xlabel("Brightness Value (0-255)")
    # plt.ylabel("Frequency")
    
    
    # print("\n📊 Interpretation:")
    # print(f"- FSB (Face Skin Brightness): {results['fsb']:.1f}/255")
    # print(f"- BIM (Brightness Information Metric): {results['bim']:.2f}")
    
    
    # Add FSB and optimal range markers
    # plt.axvline(x=results['fsb'], color='r', linestyle='--', label=f'FSB: {results["fsb"]:.1f}')
    # plt.axvspan(160, 205, alpha=0.2, color='green', label='Optimal Range (160-205)')
    # plt.legend()
    # plt.show()
    
    # Print interpretation

    # if results['fsb'] < 160:
    #     print("  ⚠️ Image may be UNDER-exposed (FSB < 160)")
    # elif results['fsb'] > 205:
    #     print("  ⚠️ Image may be OVER-exposed (FSB > 205)")
    # else:
    #     print("  ✅ Image is in optimal brightness range (160-205)")

# ==========================================
# 5. NOTEBOOK INTERFACE
# ==========================================
def analyze_fsb_lighting(img=None,img_path=None):
    """Complete analysis workflow"""
    if img is None and img_path is not None:
        processed_img = cv2.imread(img_path)
    elif isinstance(img, Image.Image):
        processed_img = pil_to_cv2(img)
    elif isinstance(img, np.ndarray):
        processed_img = img.copy()
    else:
        raise ValueError("You must provide either img or img_path.")
    try:
        results = process_fsb_lighting_face_image(processed_img)
        display_results(results)
        return results
    except Exception as e:
        print(f"❌ Analysis failed: {e}")
        # raise


def get_fsb_bounds(race, gender, lower_percentile=5, upper_percentile=95):
    stats = demographic_fsb_stats.get((race.lower(), gender.lower()))
    if not stats:
        raise ValueError(f"Demographic {(race, gender)} not found.")
    
    mean, std = stats["mean"], stats["std"]
    
    # Get z-scores for percentiles (using norm.ppf for standard normal)
    z_lower = scipy.stats.norm.ppf(lower_percentile / 100)
    z_upper = scipy.stats.norm.ppf(upper_percentile / 100)
    
    # print(z_lower, z_upper)
    
    # Your original equation
    lower_bound = mean + z_lower * std
    upper_bound = mean + z_upper * std
    
    return (lower_bound, upper_bound)

# get_fsb_bounds("black", "male")


✅ Python path configured:
→ /home/nessessence/mnt_tl_vision16/home/nessessence/uul/FaceBrightness
→ /home/nessessence/mnt_tl_vision16/home/nessessence/uul/FaceBrightness/face_parsing
✓ Importing from: /home/nessessence/mnt_tl_vision16/home/nessessence/uul/FaceBrightness/face_parsing/test.py
✓ All imports successful!


In [92]:
data_info = {
    # 'osama': {'race': 'black', 'gender': 'male', 'seen': False},
    # 'honer': {'race': 'white', 'gender': 'male', 'seen': False},
    
    'nivola': {'race': 'white', 'gender': 'male','unseen': True},
    'leowoodal': {'race': 'white', 'gender': 'male', 'unseen': True},
    'starkey': {'race': 'white', 'gender': 'male', 'unseen': True},
    
    'asante': {'race': 'black', 'gender': 'male', 'unseen': True},
    'apierre': {'race': 'black', 'gender': 'male', 'unseen': True},
    'skyhblack': {'race': 'black', 'gender': 'male', 'unseen': True},

    'reese': {'race': 'black', 'gender': 'female', 'unseen': True},
    'sophiewilde':{'race': 'black', 'gender': 'female', 'unseen': True},
    'edebiri':{'race': 'black', 'gender': 'female', 'unseen': True},
    
    'earle': {'race': 'white', 'gender': 'female', 'unseen': True},
    'mmadison': {'race': 'white', 'gender': 'female', 'unseen': True},
    'nicoparker': {'race': 'white', 'gender': 'female', 'unseen': True},

    'edsheeran':{'race': 'white', 'gender': 'male', 'unseen': False, 'full_name': 'ed-sheeran'},
    'chemsworth': {'race': 'white', 'gender': 'male', 'unseen': False,'full_name': 'chris-hemsworth'},  # Chris Hemsworth
    'cevans': {'race': 'white', 'gender': 'male', 'unseen': False,'full_name': 'chris-evans'},      # Chris Evans
    # 'adriver': {'race': 'white', 'gender': 'male', 'unseen': False,'full_name':'adam-driver'},     # Adam Driver
    # 'agarfield': {'race': 'white', 'gender': 'male', 'unseen': False,'full_name':'andrew-garfield'},   # Andrew Garfield
    
    'mrobbie': {'race': 'white', 'gender': 'female', 'unseen': False, 'full_name': 'margot-robbie'},
    'aadam': {'race': 'white', 'gender': 'female', 'unseen': False,'full_name': 'anne-adam'},       # Anne Adam
    'ahathaway': {'race': 'white', 'gender': 'female', 'unseen': False,'full_name': 'anne-hathaway'}, # Anne Hathaway
    # 'ajolie': {'race': 'white', 'gender': 'female', 'unseen': False,'full_name': 'angelina-jolie'},    # Angelina Jolie
    # 'amber': {'race': 'white', 'gender': 'female', 'unseen': False,'full_name': 'amber-heard'},     # Likely Amber Heard
    
    'rihanna': {'race': 'black', 'gender': 'female', 'unseen': False, 'full_name': 'rihanna'},
    'mcarey': {'race': 'black', 'gender': 'female', 'unseen': False,'full_name':'mariah-carey'},    # Mariah Carey (black heritage)
    'octavia': {'race': 'black', 'gender': 'female', 'unseen': False,'full_name':'octavia-spencer'},   # Octavia Spencer
    # 'oprah': {'race': 'black', 'gender': 'female', 'unseen': False,'full_name':'oprah-winfrey'},     # Oprah Winfrey
    
    'obama': {'race': 'black', 'gender': 'male', 'unseen': False, 'full_name': 'barrack-obama'},
    'morganf': {'race': 'black', 'gender': 'male', 'unseen': False,'full_name':'morgan-freeman'},     # Morgan Freeman
    'drake': {'race': 'black', 'gender': 'male', 'unseen': False,'full_name':'drake'},       # Drake (mixed but usually listed as black)
    # 'idris': {'race': 'black', 'gender': 'male', 'unseen': False,'full_name':'idris-elba'},       # Idris Elba
    
}

for concept in data_info.keys():
    data_info[concept]['img_path'] = f'data_root/data/real_data/{concept}/aligned/{concept}-5-v0'
    
    
seen_concepts = []; unseen_concepts = []
for name, info in data_info.items():
    if info.get('unseen', False):
        unseen_concepts.append(name)
    else:
        seen_concepts.append(name)
print(f"Seen concepts: {seen_concepts}")
print(f"Unseen concepts: {unseen_concepts}")

Seen concepts: ['edsheeran', 'chemsworth', 'cevans', 'mrobbie', 'aadam', 'ahathaway', 'rihanna', 'mcarey', 'octavia', 'obama', 'morganf', 'drake']
Unseen concepts: ['nivola', 'leowoodal', 'starkey', 'asante', 'apierre', 'skyhblack', 'reese', 'sophiewilde', 'edebiri', 'earle', 'mmadison', 'nicoparker']


# Experiment

In [93]:
data_root = "data_root/data/real_data/FFHQ"
image_paths = natsorted(get_image_paths(data_root))

seed = 123
rng = np.random.RandomState(seed)  # Local random state
rng.shuffle(image_paths)  # Shuffles in-place

In [94]:
n = 100
dst_folder = "data_root/data/real_data/morph_ffhq/anchor_images"
os.makedirs(dst_folder, exist_ok=True)
sampled_image_paths = image_paths[:n]
for img_path in sampled_image_paths:
    # results = analyze_fsb_lighting(img_path=img_path)
    shutil.copy(img_path, dst_folder)


In [110]:
# test_x = []
def prepare_morph_folder(start_idx=None,end_idx=None):
    root_output_img_dir = "data_root/data/real_data/morph_ffhq/n30_v0"
    anchor_dir_img_path = "data_root/data/real_data/morph_ffhq/anchor_images"
    for seen_concept,unseen_concept in zip(seen_concepts, unseen_concepts):
        # print(f"Processing {seen_concept} and {unseen_concept}...")
        seen_info = data_info[seen_concept]
        unseen_info = data_info[unseen_concept]
        
        seen_dir_img_path = seen_info['img_path']
        unseen_dir_img_path = unseen_info['img_path']
        

        seen_image_paths = natsorted(get_image_paths(seen_dir_img_path))
        unseen_image_paths = natsorted(get_image_paths(unseen_dir_img_path))
        # anchor_image_paths = natsorted(get_image_paths(anchor_dir_img_path))
        anchor_image_paths = sampled_image_paths 
        
        rng = np.random.RandomState(seed)  # Local random state
        for i,anchor_img_path in enumerate(anchor_image_paths):
            output_seen_dir_img_path = os.path.join(root_output_img_dir, f"{seen_concept}", f"{i}")
            output_unseen_dir_img_path = os.path.join(root_output_img_dir, f"{unseen_concept}", f"{i}")
            os.makedirs(output_seen_dir_img_path, exist_ok=True)
            os.makedirs(output_unseen_dir_img_path, exist_ok=True)
            
            sampled_seen_img_path = rng.choice(seen_image_paths)  
            sampled_unseen_img_path = rng.choice(unseen_image_paths)  


            if (start_idx is not None and end_idx is not None) and not (start_idx <= i < end_idx):
                print(f"Skipping Anchor:{i}")
                continue
            # seen
            output_target_seen_img_dir = os.path.join(output_seen_dir_img_path, 'img0')
            output_anchor_seen_img_dir = os.path.join(output_seen_dir_img_path, 'img1')
            os.makedirs(output_target_seen_img_dir, exist_ok=True)
            os.makedirs(output_anchor_seen_img_dir, exist_ok=True)
            shutil.copy(sampled_seen_img_path, output_target_seen_img_dir)
            shutil.copy(anchor_img_path, output_anchor_seen_img_dir)
            # unseen
            output_target_unseen_img_dir = os.path.join(output_unseen_dir_img_path, 'img0')
            output_anchor_unseen_img_dir = os.path.join(output_unseen_dir_img_path, 'img1')
            os.makedirs(output_target_unseen_img_dir, exist_ok=True)
            os.makedirs(output_anchor_unseen_img_dir, exist_ok=True)
            shutil.copy(sampled_unseen_img_path, output_target_unseen_img_dir)
            shutil.copy(anchor_img_path, output_anchor_unseen_img_dir)
            
            
            
def get_image_paths_from_pair_dir(script_dir,start_idx=None,end_idx=None):
    """
    Retrieves seen and unseen image paths for each folder in the script directory.
    
    Args:
        script_dir (str): Path to the directory containing the seen-unseen pairs folders
        
    Returns:
        dict: A dictionary where keys are folder names and values are dictionaries with
              'seen' and 'unseen' image paths
    """
    image_paths = {}
    
    for concept in os.listdir(script_dir):
        concept_path = os.path.join(script_dir, concept)

        for _id in natsorted(os.listdir(concept_path)):
            folder_path = os.path.join(concept_path, _id)
        
            # hack: 
            # if int(_id) < 30: continue
            
            
            if (start_idx is not None and end_idx is not None) and not (start_idx <= int(_id) < end_idx):
                print(f"Skipping Anchor:{int(_id)}")
                continue

            # Skip if not a directory
            if not os.path.isdir(folder_path):
                continue
                
            target_dir = os.path.join(folder_path, 'img0')
            anchor_dir = os.path.join(folder_path, 'img1')
            
            # Get target image path (assuming exactly one image in target directory)
            target_images = [f for f in os.listdir(target_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
            if len(target_images) == 1:
                target_path = os.path.join(target_dir, target_images[0])
            else:
                raise ValueError(f"Expected exactly one image in {target_dir}, found {len(target_images)}")
            
            # Get anchor image path (assuming exactly one image in anchor directory)
            anchor_images = [f for f in os.listdir(anchor_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
            if len(anchor_images) == 1:
                anchor_path = os.path.join(anchor_dir, anchor_images[0])
            else:
                raise ValueError(f"Expected exactly one image in {anchor_dir}, found {len(anchor_images)}")
            
            image_paths[f"{concept}_{_id}"] = {
                'target': target_path,
                'anchor': anchor_path
            }
            
            script = f"""
            python3 main.py --image_path_0 "{target_path}" --image_path_1 "{anchor_path}" --output_path "{os.path.join(folder_path, 'morphed')}" \\"""
            script += f"""
                --prompt_0 'a photo of a person' --prompt_1 'a photo of a person' \\"""

            script += f"""
                --save_lora_dir "{os.path.join(folder_path, 'lora')}" \\
                --use_adain --use_reschedule --save_inter --num_frames 20
            """
            print(f"Generated script for {concept}_{_id}: {script}")
            image_paths[f"{concept}_{_id}"]['script'] = script
    return image_paths





In [ ]:
# prepare_morph_folder(start_idx=0, end_idx=30) # this is might used anchor_image_paths = natsorted(get_image_paths(anchor_dir_img_path)) ... not sure
# prepare_morph_folder(start_idx=30, end_idx=50)
prepare_morph_folder(start_idx=50, end_idx=100)

Skipping Anchor:0
Skipping Anchor:1
Skipping Anchor:2
Skipping Anchor:3
Skipping Anchor:4
Skipping Anchor:5
Skipping Anchor:6
Skipping Anchor:7
Skipping Anchor:8
Skipping Anchor:9
Skipping Anchor:10
Skipping Anchor:11
Skipping Anchor:12
Skipping Anchor:13
Skipping Anchor:14
Skipping Anchor:15
Skipping Anchor:16
Skipping Anchor:17
Skipping Anchor:18
Skipping Anchor:19
Skipping Anchor:20
Skipping Anchor:21
Skipping Anchor:22
Skipping Anchor:23
Skipping Anchor:24
Skipping Anchor:25
Skipping Anchor:26
Skipping Anchor:27
Skipping Anchor:28
Skipping Anchor:29
Skipping Anchor:30
Skipping Anchor:31
Skipping Anchor:32
Skipping Anchor:33
Skipping Anchor:34
Skipping Anchor:35
Skipping Anchor:36
Skipping Anchor:37
Skipping Anchor:38
Skipping Anchor:39
Skipping Anchor:40
Skipping Anchor:41
Skipping Anchor:42
Skipping Anchor:43
Skipping Anchor:44
Skipping Anchor:45
Skipping Anchor:46
Skipping Anchor:47
Skipping Anchor:48
Skipping Anchor:49
Skipping Anchor:0
Skipping Anchor:1
Skipping Anchor:2
Skippi

In [121]:
pair_dir = 'data_root/data/real_data/morph_ffhq/n30_v0'
# morph_scripts = get_image_paths_from_pair_dir(pair_dir,start_idx=0,end_idx=30)
# morph_scripts = get_image_paths_from_pair_dir(pair_dir,start_idx=30,end_idx=50)
morph_scripts = get_image_paths_from_pair_dir(pair_dir,start_idx=50,end_idx=100)

Skipping Anchor:0
Skipping Anchor:1
Skipping Anchor:2
Skipping Anchor:3
Skipping Anchor:4
Skipping Anchor:5
Skipping Anchor:6
Skipping Anchor:7
Skipping Anchor:8
Skipping Anchor:9
Skipping Anchor:10
Skipping Anchor:11
Skipping Anchor:12
Skipping Anchor:13
Skipping Anchor:14
Skipping Anchor:15
Skipping Anchor:16
Skipping Anchor:17
Skipping Anchor:18
Skipping Anchor:19
Skipping Anchor:20
Skipping Anchor:21
Skipping Anchor:22
Skipping Anchor:23
Skipping Anchor:24
Skipping Anchor:25
Skipping Anchor:26
Skipping Anchor:27
Skipping Anchor:28
Skipping Anchor:29
Skipping Anchor:30
Skipping Anchor:31
Skipping Anchor:32
Skipping Anchor:33
Skipping Anchor:34
Skipping Anchor:35
Skipping Anchor:36
Skipping Anchor:37
Skipping Anchor:38
Skipping Anchor:39
Skipping Anchor:40
Skipping Anchor:41
Skipping Anchor:42
Skipping Anchor:43
Skipping Anchor:44
Skipping Anchor:45
Skipping Anchor:46
Skipping Anchor:47
Skipping Anchor:48
Skipping Anchor:49
Generated script for octavia_50: 
            python3 main.p

In [123]:
print(len(morph_scripts))

1200


In [ ]:
# distributed to each machine
print('')
n = 7
len_ = 150
for idx, pair in enumerate(list(morph_scripts.keys())[n*len_:n*len_+len_]):
    print(f"echo \"{n*len_+idx} {pair}\"")
    print(f"{morph_scripts[pair]['script']}")

480

In [ ]:
morph_scripts

In [53]:
# continue from 30 to 30 + 20
# debug note: ordering this id

root_output_img_dir = "data_root/data/real_data/morph_ffhq/n30_v0"
# anchor_dir_img_path = "data_root/data/real_data/morph_ffhq/anchor_images"
for seen_concept,unseen_concept in zip(seen_concepts, unseen_concepts):
    # print(f"Processing {seen_concept} and {unseen_concept}...")
    seen_info = data_info[seen_concept]
    unseen_info = data_info[unseen_concept]
    
    seen_dir_img_path = seen_info['img_path']
    unseen_dir_img_path = unseen_info['img_path']
    


    seen_image_paths = natsorted(get_image_paths(seen_dir_img_path))
    unseen_image_paths = natsorted(get_image_paths(unseen_dir_img_path))
    # anchor_image_paths = natsorted(get_image_paths(anchor_dir_img_path))
    anchor_image_paths = sampled_image_paths  # continue from 30 to 30 + 20

    rng = np.random.RandomState(seed)  # Local random state
    for i,anchor_img_path in enumerate(anchor_image_paths):
        # i = i + 30  # continue from 30 to 30 + 20
        output_seen_dir_img_path = os.path.join(root_output_img_dir, f"{seen_concept}", f"{i}")
        output_unseen_dir_img_path = os.path.join(root_output_img_dir, f"{unseen_concept}", f"{i}")
        os.makedirs(output_seen_dir_img_path, exist_ok=True)
        os.makedirs(output_unseen_dir_img_path, exist_ok=True)
        
        sampled_seen_img_path = rng.choice(seen_image_paths)  
        sampled_unseen_img_path = rng.choice(unseen_image_paths)  

        if i < 30:
            print(f"Skipping {i} for {seen_concept} and {unseen_concept} (less than 30)")
            continue
        
        # seen
        output_target_seen_img_dir = os.path.join(output_seen_dir_img_path, 'img0')
        output_anchor_seen_img_dir = os.path.join(output_seen_dir_img_path, 'img1')
        os.makedirs(output_target_seen_img_dir, exist_ok=True)
        os.makedirs(output_anchor_seen_img_dir, exist_ok=True)
        shutil.copy(sampled_seen_img_path, output_target_seen_img_dir)
        shutil.copy(anchor_img_path, output_anchor_seen_img_dir)
        # unseen
        output_target_unseen_img_dir = os.path.join(output_unseen_dir_img_path, 'img0')
        output_anchor_unseen_img_dir = os.path.join(output_unseen_dir_img_path, 'img1')
        os.makedirs(output_target_unseen_img_dir, exist_ok=True)
        os.makedirs(output_anchor_unseen_img_dir, exist_ok=True)
        shutil.copy(sampled_unseen_img_path, output_target_unseen_img_dir)
        shutil.copy(anchor_img_path, output_anchor_unseen_img_dir)


Skipping 0 for edsheeran and nivola (less than 30)
Skipping 1 for edsheeran and nivola (less than 30)
Skipping 2 for edsheeran and nivola (less than 30)
Skipping 3 for edsheeran and nivola (less than 30)
Skipping 4 for edsheeran and nivola (less than 30)
Skipping 5 for edsheeran and nivola (less than 30)
Skipping 6 for edsheeran and nivola (less than 30)
Skipping 7 for edsheeran and nivola (less than 30)
Skipping 8 for edsheeran and nivola (less than 30)
Skipping 9 for edsheeran and nivola (less than 30)
Skipping 10 for edsheeran and nivola (less than 30)
Skipping 11 for edsheeran and nivola (less than 30)
Skipping 12 for edsheeran and nivola (less than 30)
Skipping 13 for edsheeran and nivola (less than 30)
Skipping 14 for edsheeran and nivola (less than 30)
Skipping 15 for edsheeran and nivola (less than 30)
Skipping 16 for edsheeran and nivola (less than 30)
Skipping 17 for edsheeran and nivola (less than 30)
Skipping 18 for edsheeran and nivola (less than 30)
Skipping 19 for edshee

In [ ]:
# continue from 50 to 50+50
# debug note: ordering this id

root_output_img_dir = "data_root/data/real_data/morph_ffhq/n30_v0"
# anchor_dir_img_path = "data_root/data/real_data/morph_ffhq/anchor_images"
for seen_concept,unseen_concept in zip(seen_concepts, unseen_concepts):
    # print(f"Processing {seen_concept} and {unseen_concept}...")
    seen_info = data_info[seen_concept]
    unseen_info = data_info[unseen_concept]
    
    seen_dir_img_path = seen_info['img_path']
    unseen_dir_img_path = unseen_info['img_path']
    


    seen_image_paths = natsorted(get_image_paths(seen_dir_img_path))
    unseen_image_paths = natsorted(get_image_paths(unseen_dir_img_path))
    # anchor_image_paths = natsorted(get_image_paths(anchor_dir_img_path))
    anchor_image_paths = sampled_image_paths  # continue from 30 to 30 + 20

    rng = np.random.RandomState(seed)  # Local random state
    for i,anchor_img_path in enumerate(anchor_image_paths):
        output_seen_dir_img_path = os.path.join(root_output_img_dir, f"{seen_concept}", f"{i}")
        output_unseen_dir_img_path = os.path.join(root_output_img_dir, f"{unseen_concept}", f"{i}")
        os.makedirs(output_seen_dir_img_path, exist_ok=True)
        os.makedirs(output_unseen_dir_img_path, exist_ok=True)
        
        sampled_seen_img_path = rng.choice(seen_image_paths)  
        sampled_unseen_img_path = rng.choice(unseen_image_paths)  

        if i < 50:
            print(f"Skipping {i} for {seen_concept} and {unseen_concept} (less than 30)")
            continue
        
        # seen
        output_target_seen_img_dir = os.path.join(output_seen_dir_img_path, 'img0')
        output_anchor_seen_img_dir = os.path.join(output_seen_dir_img_path, 'img1')
        os.makedirs(output_target_seen_img_dir, exist_ok=True)
        os.makedirs(output_anchor_seen_img_dir, exist_ok=True)
        shutil.copy(sampled_seen_img_path, output_target_seen_img_dir)
        shutil.copy(anchor_img_path, output_anchor_seen_img_dir)
        # unseen
        output_target_unseen_img_dir = os.path.join(output_unseen_dir_img_path, 'img0')
        output_anchor_unseen_img_dir = os.path.join(output_unseen_dir_img_path, 'img1')
        os.makedirs(output_target_unseen_img_dir, exist_ok=True)
        os.makedirs(output_anchor_unseen_img_dir, exist_ok=True)
        shutil.copy(sampled_unseen_img_path, output_target_unseen_img_dir)
        shutil.copy(anchor_img_path, output_anchor_unseen_img_dir)


In [33]:
def get_image_paths_from_pair_dir(script_dir):
    """
    Retrieves seen and unseen image paths for each folder in the script directory.
    
    Args:
        script_dir (str): Path to the directory containing the seen-unseen pairs folders
        
    Returns:
        dict: A dictionary where keys are folder names and values are dictionaries with
              'seen' and 'unseen' image paths
    """
    image_paths = {}
    
    for concept in os.listdir(script_dir):
        concept_path = os.path.join(script_dir, concept)

        for _id in natsorted(os.listdir(concept_path)):
            folder_path = os.path.join(concept_path, _id)
        
        
            # Skip if not a directory
            if not os.path.isdir(folder_path):
                continue
                
            target_dir = os.path.join(folder_path, 'img0')
            anchor_dir = os.path.join(folder_path, 'img1')
            
            # Get target image path (assuming exactly one image in target directory)
            target_images = [f for f in os.listdir(target_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
            if len(target_images) == 1:
                target_path = os.path.join(target_dir, target_images[0])
            else:
                raise ValueError(f"Expected exactly one image in {target_dir}, found {len(target_images)}")
            
            # Get anchor image path (assuming exactly one image in anchor directory)
            anchor_images = [f for f in os.listdir(anchor_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
            if len(anchor_images) == 1:
                anchor_path = os.path.join(anchor_dir, anchor_images[0])
            else:
                raise ValueError(f"Expected exactly one image in {anchor_dir}, found {len(anchor_images)}")
            
            image_paths[f"{concept}_{_id}"] = {
                'target': target_path,
                'anchor': anchor_path
            }
            
            script = f"""
            python3 main.py --image_path_0 "{target_path}" --image_path_1 "{anchor_path}" --output_path "{os.path.join(folder_path, 'morphed')}" \\"""
            script += f"""
                --prompt_0 'a photo of a person' --prompt_1 'a photo of a person' \\"""

            script += f"""
                --save_lora_dir "{os.path.join(folder_path, 'lora')}" \\
                --use_adain --use_reschedule --save_inter --num_frames 20
            """
            print(f"Generated script for {concept}_{_id}: {script}")
            image_paths[f"{concept}_{_id}"]['script'] = script
    return image_paths


pair_dir = 'data_root/data/real_data/morph_ffhq/n30_v0'

morph_scripts = get_image_paths_from_pair_dir(pair_dir)



  

Generated script for octavia_0: 
            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0/octavia/0/img0/Octavia-Spencer-2019 (1).png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0/octavia/0/img1/00493.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0/octavia/0/morphed" \
                --prompt_0 'a photo of a person' --prompt_1 'a photo of a person' \
                --save_lora_dir "data_root/data/real_data/morph_ffhq/n30_v0/octavia/0/lora" \
                --use_adain --use_reschedule --save_inter --num_frames 20
            
Generated script for octavia_1: 
            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0/octavia/1/img0/Octavia-Spencer-2019 (1).png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0/octavia/1/img1/02112.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0/octavia/1/morphed" \
                --prompt_0 'a photo of a person' --prompt_1 'a photo of a per

In [120]:
for pair in list(morph_scripts.keys())[:200]:

    print(f"echo {pair}")
    print(f"{morph_scripts[pair]['script']}")

echo octavia_30

            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0/octavia/30/img0/OS-TBT-Headshot-scaled-e1669063063542.png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0/octavia/30/img1/08554.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0/octavia/30/morphed" \
                --prompt_0 'a photo of a person' --prompt_1 'a photo of a person' \
                --save_lora_dir "data_root/data/real_data/morph_ffhq/n30_v0/octavia/30/lora" \
                --use_adain --use_reschedule --save_inter --num_frames 20
            
echo octavia_31

            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0/octavia/31/img0/octavia-spencer-1a7cf64f0ae54ed1b7350bb94638f163.png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0/octavia/31/img1/18758.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0/octavia/31/morphed" \
                --prompt_0 'a photo of a person' --prompt_1 'a ph

In [37]:
for pair in list(morph_scripts.keys())[:200]:

    print(f"echo {pair}")
    print(f"{morph_scripts[pair]['script']}")

echo octavia_0

            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0/octavia/0/img0/Octavia-Spencer-2019 (1).png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0/octavia/0/img1/00493.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0/octavia/0/morphed" \
                --prompt_0 'a photo of a person' --prompt_1 'a photo of a person' \
                --save_lora_dir "data_root/data/real_data/morph_ffhq/n30_v0/octavia/0/lora" \
                --use_adain --use_reschedule --save_inter --num_frames 20
            
echo octavia_1

            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0/octavia/1/img0/Octavia-Spencer-2019 (1).png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0/octavia/1/img1/02112.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0/octavia/1/morphed" \
                --prompt_0 'a photo of a person' --prompt_1 'a photo of a person' \
                --save_lora

In [38]:
for pair in list(morph_scripts.keys())[200:400]:

    print(f"echo {pair}")
    print(f"{morph_scripts[pair]['script']}")

echo edebiri_20

            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0/edebiri/20/img0/2200065154.png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0/edebiri/20/img1/39336.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0/edebiri/20/morphed" \
                --prompt_0 'a photo of a person' --prompt_1 'a photo of a person' \
                --save_lora_dir "data_root/data/real_data/morph_ffhq/n30_v0/edebiri/20/lora" \
                --use_adain --use_reschedule --save_inter --num_frames 20
            
echo edebiri_21

            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0/edebiri/21/img0/Ayo-Edebiri-Lead-48628344e24b406ba9a85f494d03f78f.png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0/edebiri/21/img1/40191.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0/edebiri/21/morphed" \
                --prompt_0 'a photo of a person' --prompt_1 'a photo of a person' \
       

In [39]:
for pair in list(morph_scripts.keys())[400:600]:

    print(f"echo {pair}")
    print(f"{morph_scripts[pair]['script']}")

echo aadam_10

            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0/aadam/10/img0/Amy-Adams-Nightbitch-Toronto-International-Film-Festival-090824-02-8be78b653e2a4d43b459219dd49982eb.png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0/aadam/10/img1/29995.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0/aadam/10/morphed" \
                --prompt_0 'a photo of a person' --prompt_1 'a photo of a person' \
                --save_lora_dir "data_root/data/real_data/morph_ffhq/n30_v0/aadam/10/lora" \
                --use_adain --use_reschedule --save_inter --num_frames 20
            
echo aadam_11

            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0/aadam/11/img0/GettyImages-1442136081.png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0/aadam/11/img1/30475.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0/aadam/11/morphed" \
                --prompt_0 'a photo of a person

In [40]:
for pair in list(morph_scripts.keys())[600:]:

    print(f"echo {pair}")
    print(f"{morph_scripts[pair]['script']}")

echo apierre_0

            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0/apierre/0/img0/Aaron_Pierre.png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0/apierre/0/img1/00493.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0/apierre/0/morphed" \
                --prompt_0 'a photo of a person' --prompt_1 'a photo of a person' \
                --save_lora_dir "data_root/data/real_data/morph_ffhq/n30_v0/apierre/0/lora" \
                --use_adain --use_reschedule --save_inter --num_frames 20
            
echo apierre_1

            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0/apierre/1/img0/Aaron-Pierre-1.png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0/apierre/1/img1/02112.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0/apierre/1/morphed" \
                --prompt_0 'a photo of a person' --prompt_1 'a photo of a person' \
                --save_lora_dir "data_root/data/r

In [54]:
def get_image_paths_from_pair_dir(script_dir):
    """
    Retrieves seen and unseen image paths for each folder in the script directory.
    
    Args:
        script_dir (str): Path to the directory containing the seen-unseen pairs folders
        
    Returns:
        dict: A dictionary where keys are folder names and values are dictionaries with
              'seen' and 'unseen' image paths
    """
    image_paths = {}
    
    for concept in os.listdir(script_dir):
        concept_path = os.path.join(script_dir, concept)

        for _id in natsorted(os.listdir(concept_path)):
            folder_path = os.path.join(concept_path, _id)
        
            # hack: 
            if int(_id) < 30: continue
            
            # Skip if not a directory
            if not os.path.isdir(folder_path):
                continue
                
            target_dir = os.path.join(folder_path, 'img0')
            anchor_dir = os.path.join(folder_path, 'img1')
            
            # Get target image path (assuming exactly one image in target directory)
            target_images = [f for f in os.listdir(target_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
            if len(target_images) == 1:
                target_path = os.path.join(target_dir, target_images[0])
            else:
                raise ValueError(f"Expected exactly one image in {target_dir}, found {len(target_images)}")
            
            # Get anchor image path (assuming exactly one image in anchor directory)
            anchor_images = [f for f in os.listdir(anchor_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
            if len(anchor_images) == 1:
                anchor_path = os.path.join(anchor_dir, anchor_images[0])
            else:
                raise ValueError(f"Expected exactly one image in {anchor_dir}, found {len(anchor_images)}")
            
            image_paths[f"{concept}_{_id}"] = {
                'target': target_path,
                'anchor': anchor_path
            }
            
            script = f"""
            python3 main.py --image_path_0 "{target_path}" --image_path_1 "{anchor_path}" --output_path "{os.path.join(folder_path, 'morphed')}" \\"""
            script += f"""
                --prompt_0 'a photo of a person' --prompt_1 'a photo of a person' \\"""

            script += f"""
                --save_lora_dir "{os.path.join(folder_path, 'lora')}" \\
                --use_adain --use_reschedule --save_inter --num_frames 20
            """
            print(f"Generated script for {concept}_{_id}: {script}")
            image_paths[f"{concept}_{_id}"]['script'] = script
    return image_paths


pair_dir = 'data_root/data/real_data/morph_ffhq/n30_v0_test'

morph_scripts = get_image_paths_from_pair_dir(pair_dir)


Generated script for octavia_30: 
            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0_test/octavia/30/img0/Octavia-Spencer-2019 (1).png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0_test/octavia/30/img1/08554.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0_test/octavia/30/morphed" \
                --prompt_0 'a photo of a person' --prompt_1 'a photo of a person' \
                --save_lora_dir "data_root/data/real_data/morph_ffhq/n30_v0_test/octavia/30/lora" \
                --use_adain --use_reschedule --save_inter --num_frames 20
            
Generated script for octavia_31: 
            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0_test/octavia/31/img0/Octavia-Spencer-2019 (1).png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0_test/octavia/31/img1/18758.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0_test/octavia/31/morphed" \
                --prompt_0 'a pho

In [119]:
for pair in list(morph_scripts.keys())[360:480]:
    print(f"echo {pair}")
    print(f"{morph_scripts[pair]['script']}")

echo asante_30

            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0/asante/30/img0/15599134098678.png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0/asante/30/img1/08554.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0/asante/30/morphed" \
                --prompt_0 'a photo of a person' --prompt_1 'a photo of a person' \
                --save_lora_dir "data_root/data/real_data/morph_ffhq/n30_v0/asante/30/lora" \
                --use_adain --use_reschedule --save_inter --num_frames 20
            
echo asante_31

            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0/asante/31/img0/1930351_full.png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0/asante/31/img1/18758.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0/asante/31/morphed" \
                --prompt_0 'a photo of a person' --prompt_1 'a photo of a person' \
                --save_lora_dir "data_root/data/r

In [58]:

for pair in list(morph_scripts.keys())[360:480]:
    print(f"echo {pair}")
    print(f"{morph_scripts[pair]['script']}")

echo asante_30

            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0_test/asante/30/img0/asante-blackk-1-2000-ea84783680014f3094b4078a07ee697c.png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0_test/asante/30/img1/08554.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0_test/asante/30/morphed" \
                --prompt_0 'a photo of a person' --prompt_1 'a photo of a person' \
                --save_lora_dir "data_root/data/real_data/morph_ffhq/n30_v0_test/asante/30/lora" \
                --use_adain --use_reschedule --save_inter --num_frames 20
            
echo asante_31

            python3 main.py --image_path_0 "data_root/data/real_data/morph_ffhq/n30_v0_test/asante/31/img0/1989809_full.png" --image_path_1 "data_root/data/real_data/morph_ffhq/n30_v0_test/asante/31/img1/18758.png" --output_path "data_root/data/real_data/morph_ffhq/n30_v0_test/asante/31/morphed" \
                --prompt_0 'a photo of a person' --prompt_1

# Verify Samples

In [ ]:
import lpips
perceptual_loss = lpips.LPIPS().cuda()  # Use GPU if available, otherwise use CPU
def calculate_lpips(image_path0, image_path1):
    img0 = lpips.im2tensor(lpips.load_image(image_path0)).cuda() # RGB image from [-1,1]
    img1 = lpips.im2tensor(lpips.load_image(image_path1)).cuda()
    return perceptual_loss(img0, img1)


# image_path0 = "data_root/data/real_data/morph_ffhq/anchor_images/00202.png"
# image_path1 = "data_root/data/real_data/morph_ffhq/n30_v0/aadam/0/img0/GettyImages-1442136081.png"
# lpips_value = calculate_lpips(image_path0, image_path1)
# print(f"{ lpips_value.item():.4f}")

anchor_image_paths = sampled_image_paths

unseen_image_paths = []
for concept in unseen_concepts:
    unseen_img_paths = [ os.path.join(data_info[concept]['img_path'], img_name) for img_name in os.listdir(data_info[concept]['img_path']) if img_name.endswith('.png')]
    unseen_image_paths += unseen_img_paths
seen_image_paths = []
for concept in seen_concepts:
    seen_img_paths = [ os.path.join(data_info[concept]['img_path'], img_name) for img_name in os.listdir(data_info[concept]['img_path']) if img_name.endswith('.png')]
    seen_image_paths += seen_img_paths
print(f"Anchor images: {len(anchor_image_paths)}")
print(f"Unseen images: {len(unseen_image_paths)}")
print(f"Seen images: {len(seen_image_paths)}")  


lpips_results = defaultdict(list)
for anchor_path in tqdm(anchor_image_paths, desc="Calculating LPIPS"):
    for unseen_path, seen_path in zip(unseen_image_paths, seen_image_paths):
        lpips_value = calculate_lpips(anchor_path, unseen_path)
        lpips_results["unseen"] += [lpips_value.item()]
        lpips_value = calculate_lpips(anchor_path, seen_path)
        lpips_results["seen"] += [lpips_value.item()]
        
clear_output(wait=True)
print("mean of unseen:", np.mean(lpips_results["unseen"]))
print("mean of seen:", np.mean(lpips_results["seen"]))


In [ ]:
A = np.array(lpips_results["unseen"])
B = np.array(lpips_results["seen"])

Force_normal = True
Force_Welch = False


# # Example data (replace with your actual data)
# A = np.random.uniform(0.0, 1.0, 300)  # Sample data for A
# B = np.random.uniform(0.2, 0.8, 300)   # Sample data for B

# Step 1: Check Normality (Shapiro-Wilk test)
_, p_A = stats.shapiro(A)
_, p_B = stats.shapiro(B)
normal_data = (p_A > 0.05) and (p_B > 0.05)  # If p > 0.05, assume normal

# Step 2: Check Variance (Levene's test)
_, p_var = stats.levene(A, B)
equal_variance = p_var > 0.05  # If p > 0.05, assume equal variance

# Step 3: Choose the appropriate test
if normal_data or Force_normal:
    if equal_variance and not Force_Welch:
        # Use Student's t-test
        t_stat, p_value = stats.ttest_ind(A, B)
        test_used = "Student's t-test"
    else:
        # Use Welch's t-test (unequal variance)
        t_stat, p_value = stats.ttest_ind(A, B, equal_var=False)
        test_used = "Welch's t-test"
else:
    # Use Mann-Whitney U test (non-parametric)
    u_stat, p_value = stats.mannwhitneyu(A, B)
    test_used = "Mann-Whitney U test"

# Step 4: Interpret the result
alpha = 0.10  # Significance level
print(f"Test used: {test_used}")
print(f"p-value: {p_value:.4f}")

if p_value < alpha:
    print("Conclusion: A and B are statistically different (reject H0) with p-value < 0.10")
else:
    print("Conclusion: No significant difference between A and B (fail to reject H0 with p-value >= 0.10)")